In [1]:
# determine whether this system demonstrates typical characteristics of a single node or perceptron in a neural network when resources are shared or competition exists
# perceptron = a linear decision boundary like in neural networks

In [ ]:
# Different graphs/scenarios
# A = chemical reactions of a single node
# B = chemical reactions represented as an artificial neural network (ANN) node
# C = effect of binding kinetics / ideal binding kinetics (low competition) 
# D = effect of (total) resource availability
# E = competing node (S2)
# F = antisigma (A2) / reduced competition (A2 binds S2)
# G = no competition 
# H = with competition

# Changing the code to fit a different data set:

    # 1. change how & what data is read in (i.e. alter inputs)
    # 2. modify inputs x1 and x2
    #3. make sure new dataset fits inputs

import pandas as pd
import numpy as np
import scipy.integrate
from sklearn.preprocessing import MinMaxScaler
import bokeh.io
import bokeh.plotting
from bokeh.models import LinearColorMapper, ColumnDataSource, ColorBar

# Initialize inline visualization engine
bokeh.io.output_notebook()

# Read and process figure 2c data from the provided CSV file
csv_filename = "figure2_panelc.csv"

# Exact column headers
x_column = "Cdil (mM)"
y_column = "DGtr (kcal/mol)"

df = pd.read_csv(csv_filename)

# Extract non-empty row entries safely
raw_coordinates = df[[x_column, y_column]].dropna().values

# Scale data between 0 and 1 to prevent competitive ODE integration crashes
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_coordinates = scaler.fit_transform(raw_coordinates)

my_x1 = scaled_coordinates[:, 0]  # Normalized Dilute Concentration
my_x2 = scaled_coordinates[:, 1]  # Normalized Transfer Free Energy
num_samples = len(my_x1)

In [ ]:
#how two nodes that process the same inputs affect the decision boundary - varying ct

In [ ]:
# Kinetics of a 2-node competitive biomolecular model with a shared resource pool
# In phase separation: ct = Scaffold Capacity, C1 & C2 = Condensate Mass

# fundamental equation block of neural netowrk kinetics with competition for a shared resource pool
   # Si = sigma factors (activators)
   # Ai = antisigma factors (inhibitors)
   # C = shared resource pool (RNA polymerase) that signma factors compete to bind
   # Ci = active complexes or outputs of the nodes once they bind the shared resource

def Sequestration_rhs(x,t,a1,b1,a2,b2,g1,g2,d,ct):
    S1, A1, S2, A2, C1, C2 = x
    C = ct - C1 - C2 # Strict mass balance conservation of the shared resource pool
    return np.array([
        a1 - d*S1 - g1*A1*S1 - g2*S1*C, # Accumulation rate of Free Component 1
        b1 - d*A1 - g1*A1*S1,           # Accumulation rate of Inhibitor 1
        a2 - d*S2 - g1*A2*S2 - g2*S2*C, # Accumulation rate of Free Component 2
        b2 - d*A2 - g1*A2*S2,           # Accumulation rate of Inhibitor 2
        g2*S1*C - d*C1,                 # Growth rate of Complex 1 (Condensate Phase)
        g2*S2*C - d*C2,                 # Growth rate of Complex 2 (Condensate Phase)
    ])

# Physical and Kinetic Constants
g1 = 1000.0   # Homotypic interaction parameter (High-affinity self-assembly)
g2 = 10.0     # Heterotypic interaction parameter (Scaffold-Client binding affinity)
d = 1.0       # Material turnover / loss rate
fixed_ct = 0.6 # A single fixed resource capacity for initial testing; can be varied in the loop below
    # ct = [0.2, 0.6, 1.0] # Shared resource capacities (e.g., variable scaffolding pools)

# Static background expressions for nodes 
b1 = 0.4
b2 = 0.4  

# Time parameters for ODE evaluation
    # tN = 100
t = np.linspace(0, 5, 100)
x0 = np.array([0., 0., 0., 0., 0., 0.]) # System initial conditions at t=0

# Generate continuous heatmap for competing node phase space
    # Establish a clean, continuous evaluation grid spanning from 0 to 1
N_grid = 35
grid_axis = np.linspace(0, 1, N_grid)
heatmap_matrix = np.zeros((N_grid, N_grid))

for i, x1i in enumerate(grid_axis):
    for j, x2j in enumerate(grid_axis):
        # Evaluate model across the entire geometric plane
        x = scipy.integrate.odeint(
            Sequestration_rhs, x0, t, 
            args=(x1i, b1, x2j, b2, g1, g2, d, fixed_ct)
        )
        # Store normalized Complex 1 output as our background density
        heatmap_matrix[j, i] = x[-1, 4] / fixed_ct

# Establish shared pink color scheme framework
pink_palette = bokeh.palettes.RdPu9[::-1] 
pink_mapper = LinearColorMapper(palette=pink_palette, low=0, high=1)
z_padded = np.c_[heatmap_matrix, np.zeros(N_grid)]
z_padded[-1, -1] = 1.

# Run model directly on the experimental data points to get their corresponding outputs for scatter overlay
scatter_outputs = np.zeros(num_samples)
for idx in range(num_samples):
    x = scipy.integrate.odeint(
        Sequestration_rhs, x0, t, 
        args=(my_x1[idx], b1, my_x2[idx], b2, g1, g2, d, fixed_ct)
    )
    scatter_outputs[idx] = x[-1, 4] / fixed_ct

source = ColumnDataSource(data=dict(x=my_x1, y=my_x2, out=scatter_outputs))

# Rendering the final composite figure with BNN phase space heatmap and overlaid data points
    # Initialize a single consolidated figure frame
p = bokeh.plotting.figure(
    title="Figure 2C Overlaid on Competitive BNN Phase Space",
    width=560, height=430,
    x_range=(0, 1.0), y_range=(0, 1.0),
    x_axis_label=f'Normalized Input ({x_column})',
    y_axis_label=f'Normalized Thermodynamic Output ({y_column})'
)

# Layer 1: Draw the Continuous Heatmap Image Background
p.image(
    image=[z_padded], x=0, y=0, 
    dw=1.0 * (1 + 1/N_grid), dh=1.0, 
    palette=pink_palette, alpha=0.65
)

# Layer 2: Overlay the Discrete Scatter Dots directly on top
p.scatter(
    'x', 'y', source=source, size=10, 
    fill_color={'field': 'out', 'transform': pink_mapper},
    line_color="#4A001F", line_width=1.3,
    legend_label="NPM1 In Vivo Data (GFP/mCherry)"
)

# Add a structural color bar context window
colorbar = ColorBar(color_mapper=pink_mapper, location=(0,0), width=12, title="Complex Fraction")
p.add_layout(colorbar, 'right')
p.title.text_font_style = "bold"
p.legend.location = "bottom_right"

# Display graph inside your workflow environment
bokeh.io.show(p)

In [ ]:
# Non competitive node 

In [ ]:
def Sequestration_rhs(x,t,a1,b1,a2,b2,g1,g2,d,ct):
    S1, A1, S2, A2, C1, C2 = x
    C = ct - C1 - C2 # Strict mass balance conservation of the shared resource pool
    
    return np.array([
        a1 - d*S1 - g1*A1*S1 - g2*S1*C, # Accumulation rate of Free Component 1
        b1 - d*A1 - g1*A1*S1,           # Accumulation rate of Inhibitor 1
        a2 - d*S2 - g1*A2*S2 - g2*S2*C, # Accumulation rate of Free Component 2
        b2 - d*A2 - g1*A2*S2,           # Accumulation rate of Inhibitor 2
        g2*S1*C - d*C1,                 # Growth rate of Complex 1 (Condensate Phase)
        g2*S2*C - d*C2,                 # Growth rate of Complex 2 (Condensate Phase)
    ])

# Hardcoded constants exactly matching your source architecture
g1_choices = [10, 100, 1000.]
g1_selected = g1_choices[2] # Using 1000.0 for a distinct activation limit edge
g2 = 10.0
d = 1.0
ct = 1/5

t = np.linspace(0, 5, 100)
x0 = np.array([0., 0., 0., 0., 0., 0.])

# Generate continuous heatmap for non-competitive node phase space
    # Establish a clean, continuous evaluation grid spanning from 0 to 1
N_grid = 35
grid_axis = np.linspace(0, 1, N_grid)
heatmap_matrix = np.zeros((N_grid, N_grid))

for i, x1i in enumerate(grid_axis):
    for j, x2j in enumerate(grid_axis):
        x = scipy.integrate.odeint(
            Sequestration_rhs, x0, t, 
            # EXACT POSITION MATCH: x1i is Activator (a1), x2j is Inhibitor (b1). Node 2 is 0.
            args=(x1i, x2j, 0, 0, g1_selected, g2, d, ct)
        )
        heatmap_matrix[j, i] = x.transpose()[4, -1] / ct

# Establish shared pink color scheme framework
pink_palette = bokeh.palettes.RdPu9[::-1] 
pink_mapper = LinearColorMapper(palette=pink_palette, low=0, high=1)
z_padded = np.c_[heatmap_matrix, np.zeros(N_grid)]
z_padded[-1, -1] = 1.

# Run model directly on the experimental data points to get their corresponding outputs for scatter overlay
scatter_outputs = np.zeros(num_samples)
for idx in range(num_samples):
    x = scipy.integrate.odeint(
        Sequestration_rhs, x0, t, 
        # Pass your real data exactly into the Activator and Inhibitor slots
        args=(my_x1[idx], my_x2[idx], 0, 0, g1_selected, g2, d, ct)
    )
    scatter_outputs[idx] = x.transpose()[4, -1] / ct

source = ColumnDataSource(data=dict(x=my_x1, y=my_x2, out=scatter_outputs))

# Rendering the final composite figure with BNN phase space heatmap and overlaid data points
    # Initialize a single consolidated figure frame
p = bokeh.plotting.figure(
    title="Figure 2C Overlaid on Non-Competitive Sequestration Landscape",
    width=560, height=430,
    x_range=(0, 1.0), y_range=(0, 1.0),
    x_axis_label=f'Normalized Concentration Input ({x_column})',
    y_axis_label=f'Normalized Inhibitory Energy Target ({y_column})'
)

# Layer 1: Draw the Continuous Heatmap Image Background
p.image(
    image=[z_padded], x=0, y=0, 
    dw=1.0 * (1 + 1/N_grid), dh=1.0, 
    palette=pink_palette, alpha=0.65
)

# Layer 2: Overlay the Discrete Scatter Dots directly on top
p.scatter(
    'x', 'y', source=source, size=10, 
    fill_color={'field': 'out', 'transform': pink_mapper},
    line_color="#4A001F", line_width=1.3,
    legend_label="NPM1 In Vivo Data (GFP/mCherry)"
)

# Add a structural color bar context window
colorbar = ColorBar(color_mapper=pink_mapper, location=(0,0), width=12, title="Complex Fraction")
p.add_layout(colorbar, 'right')
p.title.text_font_style = "bold"
p.legend.location = "bottom_right"

# Display graph inside workflow environment
bokeh.io.show(p)

In [ ]:
def Sequestration_rhs(x,t,a1,b1,a2,b2,g1,g2,d,ct):
    S1, A1, S2, A2, C1, C2 = x
    C = ct - C1 - C2 # Strict mass balance conservation of the shared resource pool
    
    return np.array([
        a1 - d*S1 - g1*A1*S1 - g2*S1*C, # Accumulation rate of Free Component 1
        b1 - d*A1 - g1*A1*S1,           # Accumulation rate of Inhibitor 1
        a2 - d*S2 - g1*A2*S2 - g2*S2*C, # Accumulation rate of Free Component 2
        b2 - d*A2 - g1*A2*S2,           # Accumulation rate of Inhibitor 2
        g2*S1*C - d*C1,                 # Growth rate of Complex 1 (Condensate Phase)
        g2*S2*C - d*C2,                 # Growth rate of Complex 2 (Condensate Phase)
    ])

# Hardcoded constants exactly matching your source architecture
g1_choices = 100
g2 = 10.0
d = 1.0
ct = 1/5
a2_choices = [0, 0.5, 1]
a2_selected = a2_choices[2] 
t = np.linspace(0, 5, 100)
x0 = np.array([0., 0., 0., 0., 0., 0.])

# Generate continuous heatmap for non-competitive node phase space
    # Establish a clean, continuous evaluation grid spanning from 0 to 1
N_grid = 35
grid_axis = np.linspace(0, 1, N_grid)
heatmap_matrix = np.zeros((N_grid, N_grid))

for i, x1i in enumerate(grid_axis):
    for j, x2j in enumerate(grid_axis):
        x = scipy.integrate.odeint(
            Sequestration_rhs, x0, t, 
            # Position-matched: x1i is a1, x2j is b1, a2_selected sits in the 3rd slot
            args=(x1i, x2j, a2_selected, b2, g1, g2, d, ct)
        )
        # Store Complex 1 fraction (index 4) exactly like your output2 variable
        heatmap_matrix[j, i] = x.transpose()[4, -1] / ct

# Establish shared pink color scheme framework
pink_palette = bokeh.palettes.RdPu9[::-1] 
pink_mapper = LinearColorMapper(palette=pink_palette, low=0, high=1)
z_padded = np.c_[heatmap_matrix, np.zeros(N_grid)]
z_padded[-1, -1] = 1.

# Run model directly on the experimental data points to get their corresponding outputs for scatter overlay
scatter_outputs = np.zeros(num_samples)
for idx in range(num_samples):
    x = scipy.integrate.odeint(
        Sequestration_rhs, x0, t, 
        # FIXED: Position arguments reduced to 8. Kept a2_selected synchronized!
        args=(my_x1[idx], my_x2[idx], a2_selected, b2, g1, g2, d, ct)
    )
    scatter_outputs[idx] = x.transpose()[4, -1] / ct

source = ColumnDataSource(data=dict(x=my_x1, y=my_x2, out=scatter_outputs))

# Rendering the final composite figure with BNN phase space heatmap and overlaid data points
    # Initialize a single consolidated figure frame
p = bokeh.plotting.figure(
    title=f"Figure 2C Overlaid on Competitive Sequestration Landscape with Hypothetical Competitor",
    width=560, height=430,
    x_range=(0, 1.0), y_range=(0, 1.0),
    x_axis_label='Normalized Activator Input: Concentration [Cdil (mM)]',
    y_axis_label='Normalized Inhibitor Input: Energy [DGtr (kcal/mol)]'
)

# Layer 1: Draw the Continuous Heatmap Image Background
p.image(
    image=[z_padded], x=0, y=0, 
    dw=1.0 * (1 + 1/N_grid), dh=1.0, 
    palette=pink_palette, alpha=0.65
)

# Layer 2: Overlay the Discrete Scatter Dots directly on top
p.scatter(
    'x', 'y', source=source, size=10, 
    fill_color={'field': 'out', 'transform': pink_mapper},
    line_color="#4A001F", line_width=1.3,
    legend_label="NPM1 In Vivo Data (GFP/mCherry)"
)

# Add a structural color bar context window
colorbar = ColorBar(color_mapper=pink_mapper, location=(0,0), width=12, title="Complex Fraction")
p.add_layout(colorbar, 'right')
p.title.text_font_style = "bold"
p.legend.location = "bottom_right"

# Display graph inside workflow environment
bokeh.io.show(p)

In [ ]:
def Sequestration_rhs(x,t,a1,b1,a2,b2,g1,g2,d,ct):
    S1, A1, S2, A2, C1, C2 = x
    C = ct - C1 - C2 # Strict mass balance conservation of the shared resource pool
    
    return np.array([
        a1 - d*S1 - g1*A1*S1 - g2*S1*C, # Accumulation rate of Free Component 1
        b1 - d*A1 - g1*A1*S1,           # Accumulation rate of Inhibitor 1
        a2 - d*S2 - g1*A2*S2 - g2*S2*C, # Accumulation rate of Free Component 2
        b2 - d*A2 - g1*A2*S2,           # Accumulation rate of Inhibitor 2
        g2*S1*C - d*C1,                 # Growth rate of Complex 1 (Condensate Phase)
        g2*S2*C - d*C2,                 # Growth rate of Complex 2 (Condensate Phase)
    ])

# Hardcoded constants exactly matching your source architecture
g1_choices = [10, 100, 1000.]
g1_selected = g1_choices[2] # Using 1000.0 for a distinct activation limit edge
g2 = 10.0
d = 1.0
ct = 1/5

t = np.linspace(0, 5, 100)
x0 = np.array([0., 0., 0., 0., 0., 0.])

# Generate condensate phase diagram background
N_grid = 35
grid_axis = np.linspace(0, 1, N_grid)
heatmap_matrix = np.zeros((N_grid, N_grid))

phase_separation_threshold = 0.4 

for i, x1i in enumerate(grid_axis):
    for j, x2j in enumerate(grid_axis):
        x = scipy.integrate.odeint(
            Sequestration_rhs, x0, t, 
            # EXACT POSITION MATCH: x1i is Activator (a1), x2j is Inhibitor (b1). Node 2 is 0.
            args=(x1i, x2j, 0, 0, g1_selected, g2, d, ct)
        )
        heatmap_matrix[j, i] = x.transpose()[4, -1] / ct

# Establish shared pink color scheme framework
pink_palette = bokeh.palettes.RdPu9[::-1] 
pink_mapper = LinearColorMapper(palette=pink_palette, low=0, high=1)
z_padded = np.c_[heatmap_matrix, np.zeros(N_grid)]
z_padded[-1, -1] = 1.

# Run model directly on the experimental data points to get their corresponding outputs for scatter overlay
scatter_outputs = np.zeros(num_samples)
for idx in range(num_samples):
    x = scipy.integrate.odeint(
        Sequestration_rhs, x0, t, 
        # Pass your real data exactly into the Activator and Inhibitor slots
        args=(my_x1[idx], my_x2[idx], 0, 0, g1_selected, g2, d, ct)
    )
    scatter_outputs[idx] = x.transpose()[4, -1] / ct

source = ColumnDataSource(data=dict(x=my_x1, y=my_x2, out=scatter_outputs))

# Rendering the final composite figure with BNN phase space heatmap and overlaid data points
    # Initialize a single consolidated figure frame
p = bokeh.plotting.figure(
    title="Figure 2C Overlaid on Non-Competitive Sequestration Landscape",
    width=560, height=430,
    x_range=(0, 1.0), y_range=(0, 1.0),
    x_axis_label=f'Normalized Concentration Input ({x_column})',
    y_axis_label=f'Normalized Inhibitory Energy Target ({y_column})'
)

# Layer 1: Draw the Continuous Heatmap Image Background
p.image(
    image=[z_padded], x=0, y=0, 
    dw=1.0 * (1 + 1/N_grid), dh=1.0, 
    palette=pink_palette, alpha=0.65
)

# Layer 2: Overlay the Discrete Scatter Dots directly on top
p.scatter(
    'x', 'y', source=source, size=10, 
    fill_color={'field': 'out', 'transform': pink_mapper},
    line_color="#4A001F", line_width=1.3,
    legend_label="NPM1 In Vivo Data (GFP/mCherry)"
)

# Add a structural color bar context window
colorbar = ColorBar(color_mapper=pink_mapper, location=(0,0), width=12, title="Complex Fraction")
p.add_layout(colorbar, 'right')
p.title.text_font_style = "bold"
p.legend.location = "bottom_right"

# Display graph inside workflow environment
bokeh.io.show(p)